In [118]:
import torch

import triton
import triton.language as tl

@triton.jit
def _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v,
                             offset_y, dtype: tl.dtype, start_m, qk_scale,
                             block_m: tl.constexpr, hidden_dim: tl.constexpr, block_n: tl.constexpr, stage: tl.constexpr,
                             offs_m: tl.constexpr, offs_n: tl.constexpr, n_ctx: tl.constexpr, non_mask: tl.constexpr, warp_specialize: tl.constexpr):
    # print(f"Stage : {stage}")
    if stage == 1:
        lo, hi = 0, start_m*block_m
    elif stage == 2:
        lo, hi = start_m*block_m, (start_m+1)*block_m
    else:
        lo, hi = 0, n_ctx

    if non_mask:
        lo, hi = 0, (start_m+1)*block_m

    offsetk_y = offset_y + lo
    offsetv_y = offset_y + lo
    for start_n in tl.range(lo, hi, block_n, warp_specialize=warp_specialize):
        # print(f"start_n : {start_n} : offset of k [{offsetk_y},0]")
        k = tl.trans(desc_k.load([offsetk_y, 0]))
        qk = tl.dot(q, k) * qk_scale
        if stage == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            qk = qk  + tl.where(mask, 0, -1.0e6)
        m_ij = tl.maximum(m_i, tl.max(qk, 1))
        qk -= m_ij[:, None]
        p = tl.math.exp(qk)
        # -- compute correction factor
        alpha = tl.math.exp(m_i - m_ij)
        l_ij = tl.sum(p, 1)
        acc = acc * alpha[:, None]

        # print(f"Offset of v [0, {offsetv_y}]")
        v = desc_v.load([offsetv_y, 0])
        p = p.to(dtype)
        acc += tl.dot(p, v)
        l_i = l_i * alpha + l_ij
        m_i = m_ij
        offsetk_y += block_n
        offsetv_y += block_n
    return acc, l_i, m_i

@triton.autotune(
    configs=[
        triton.Config({'block_m':32, 'block_n':16}, num_warps=4, num_stages=1),
        triton.Config({'block_m':16, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':32, 'block_n':32}, num_warps=4, num_stages=1),
        triton.Config({'block_m':16, 'block_n':16}, num_warps=4, num_stages=1)
    ],
    key=['n_ctx', 'hidden_dim'],   # runtime-dependent shapes
)
@triton.jit
def _attention_forward(sm_scale, max_tensor, softmax_dem, batch, num_heads, n_ctx, desc_q, desc_k, desc_v, desc_o, lower_precision: tl.constexpr,
                       hidden_dim: tl.constexpr, mask_region: tl.constexpr, warp_specialize: tl.constexpr,
                       block_m: tl.constexpr, block_n: tl.constexpr):
    dtype: tl.dtype = tl.float16 if lower_precision else tl.float32
    assert block_n <= hidden_dim
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    batch_idx = off_hz // num_heads
    head_idx = off_hz % num_heads
    # print(f"start_m : {start_m}, off_h : {off_h}")
    y_dim = num_heads * n_ctx * batch
    desc_q = tl.make_tensor_descriptor(desc_q, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    desc_v = tl.make_tensor_descriptor(desc_v, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                         block_shape=[block_n, hidden_dim])
    desc_k = tl.make_tensor_descriptor(desc_k, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_n, hidden_dim])
    desc_o = tl.make_tensor_descriptor(desc_o, shape=[y_dim, hidden_dim], strides=[hidden_dim, 1],
                                     block_shape=[block_m, hidden_dim])
    offset_y = batch_idx*num_heads*n_ctx + head_idx*n_ctx
    # print(f"offset_y : {offset_y}")
    qo_offset_y = offset_y + start_m*block_m
    # print(f"qo_offset_y : {qo_offset_y}")
    offs_m = start_m*block_m + tl.arange(0, block_m)
    offs_n = tl.arange(0, block_n)
    # print(f"offs_m : {offs_m}")
    # print(f"offs_n : {offs_n}")
    # initialize pointer to m and l
    m_i = tl.zeros([block_m], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([block_m], dtype=tl.float32) + 1.0
    acc = tl.zeros([block_m, hidden_dim], dtype=tl.float32)
    # load scales
    qk_scale = sm_scale
    # qk_scale *= 1.44269504 #1/log(2)
    q = desc_q.load([qo_offset_y,0])
    # print(f"q load : {[qo_offset_y, 0]}")
    if mask_region:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 1, offs_m, offs_n, n_ctx, False, warp_specialize)
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, False, warp_specialize)
    else:
        acc, l_i, m_i = _attention_forward_inner_mask(acc, l_i, m_i, q, desc_k, desc_v, offset_y, dtype, start_m, qk_scale, block_m, hidden_dim, block_n, 2, offs_m, offs_n, n_ctx, True, warp_specialize)
    m_i += tl.math.log(l_i)
    acc = acc / l_i[:, None]
    off_hz = batch_idx*num_heads*n_ctx + head_idx*n_ctx
    m_ptrs = max_tensor + off_hz + offs_m
    sft_dem_ptrs = softmax_dem + off_hz + offs_m
    tl.store(m_ptrs, m_i)
    tl.store(sft_dem_ptrs, l_i)
    desc_o.store([qo_offset_y, 0], acc.to(dtype))

In [119]:
def forward_call(q, k, v):
  warp_specialize = True
  batch, num_heads, n_ctx, hidden_dim = q.shape[0], q.shape[1], q.shape[2], q.shape[3]
  sm_scale = 1.0 / (hidden_dim ** 0.5)

  o = torch.empty_like(q)
  M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
  sft_d = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)

  grid_fwd = lambda META: (
      triton.cdiv(n_ctx, META['block_m']),
      num_heads * batch,
      1
  )
  if q.dtype == torch.float16:
    lower_precision = True
  else:
    lower_precision = False
  # grid = (n_ctx//block_m, num_heads*batch, 1)
  # print(f"Grid : {grid}")

  # Attention forward
  # mask region
  _attention_forward[grid_fwd](sm_scale, M, sft_d, batch, num_heads, n_ctx,
                  q, k, v, o, lower_precision,
                  hidden_dim, True, warp_specialize,)
  sft_d = sft_d.unsqueeze(-1)
  # print(o.shape)
  # print(sft_d.shape)
  # print(torch.tensor([torch.sum(o[0][0][i][:]) for i in range(32)]))
  # print(sft_d[0][0])
  # o = o / sft_d
  return o

def compare_outputs(ref, custom, label=""):
    diff = (ref - custom).abs()
    cos  = torch.nn.functional.cosine_similarity(
               ref.flatten(), custom.flatten(), dim=0)
    print(f"{label}")
    print(f"  Max abs error : {diff.max().item():.6f}")
    print(f"  Mean abs error: {diff.mean().item():.6f}")
    print(f"  Cosine sim    : {cos.item():.8f}")
    print(f"  Rel error     : {(diff / (ref.abs() + 1e-8)).mean().item():.6f}")

q = torch.randn([1,1,16,16], device='cuda', dtype=torch.float16)
k = torch.randn([1,1,16,16], device='cuda', dtype=torch.float16)
v = torch.randn([1,1,16,16], device='cuda', dtype=torch.float16)
q32 = q.float()
k32 = k.float()
v32 = v.float()
torch_attention = torch.nn.functional.scaled_dot_product_attention
output_torch = torch_attention(q, k, v, is_causal=True)
output_custom_triton_16 = forward_call(q, k, v)
compare_outputs(output_torch, output_custom_triton_16, "triton vs pytorch(fp16)")
output_torch_32 = torch_attention(q, k, v, is_causal=True)
output_custom_triton_32 = forward_call(q32, k32, v32)
compare_outputs(output_torch_32, output_custom_triton_32, "triton vs pytorch(fp32)")
print(output_custom_triton_32-output_torch_32)




triton vs pytorch(fp16)
  Max abs error : 0.000000
  Mean abs error: 0.000000
  Cosine sim    : 0.99951172
  Rel error     : 0.000000
triton vs pytorch(fp32)
  Max abs error : 0.000453
  Mean abs error: 0.000072
  Cosine sim    : 0.99986756
  Rel error     : 0.000578
tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
            0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
            0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
            0.0000e+00],
          [-3.0726e-05,  5.8979e-05, -3.3349e-05, -1.0085e-04, -1.4919e-04,
            1.6913e-05,  9.8914e-05, -1.8430e-04, -8.4341e-06,  4.1962e-05,
           -8.7053e-05,  2.1052e-04, -8.9377e-05,  1.8775e-05, -4.9472e-06,
            7.4370e-05],
          [ 4.7326e-05,  6.3479e-05,  2.7031e-05, -1.1045e-04,  1.6069e-04,
            3.0398e-05,  6.0409e-05,  3.6108e-04,  1.9222e-06,  1.9908e-04,
            1.2219e-05, -6.1989e-05,  6.6534e-05, -2.4050e-05,  5.2065e-05

In [57]:
input = torch.randn([1,1,5,5])
print(input)
print(torch.softmax(input, dim=-1))

tensor([[[[ 0.0626,  0.4746, -0.7744,  1.6908,  0.0126],
          [ 0.4580,  0.4495, -1.2912, -0.4015, -0.5222],
          [-0.7477, -1.6901,  0.6193,  0.0243, -0.7007],
          [ 1.9838, -1.5521,  2.0412, -1.3918, -1.3661],
          [-1.7924,  0.2518, -0.0239,  1.1072, -0.9215]]]])
tensor([[[[0.1112, 0.1680, 0.0482, 0.5668, 0.1058],
          [0.3374, 0.3345, 0.0587, 0.1428, 0.1266],
          [0.1173, 0.0457, 0.4602, 0.2538, 0.1229],
          [0.4635, 0.0135, 0.4909, 0.0159, 0.0163],
          [0.0285, 0.2198, 0.1668, 0.5170, 0.0680]]]])
